# Predictive Financial Time-Series Forecasting Using S&P 500 Data
**End-to-End Predictive Analytics / KDD Project**

**Problem statement:** Predict the next trading day's closing price using historical S&P 500 stock data.

**Objective:** Clean and understand financial data, engineer features, train predictive models, evaluate them chronologically, and interpret the results.

This is an educational research project, not financial advice.

## 1. Obtain the dataset

Recommended source: [Kaggle — S&P 500 stock data](https://www.kaggle.com/camnugent/sandp500). Download the CSV (commonly `all_stocks_5yr.csv`) after signing in, then place it in the `dataset/` folder. Typical columns are `date, open, high, low, close, volume, Name`.

Alternative: download daily historical data for a ticker such as SPY from [Yahoo Finance](https://finance.yahoo.com/) as CSV.

Check the dataset's license before redistributing it. If the dataset has many companies, this notebook selects one ticker/company for the experiment.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = Path("dataset/all_stocks_5yr.csv")  # Change this to your CSV file path
TICKER_FILTER = None  # e.g. "AAPL"; None selects the first ticker if multiple companies exist
TEST_FRACTION = 0.15
VALIDATION_FRACTION = 0.15
RANDOM_STATE = 42

print("Expected CSV:", DATA_PATH.resolve())

## 2. Load and standardize data

In [ ]:
if not DATA_PATH.exists():
    csvs = list(Path("dataset").glob("*.csv")) + list(Path(".").glob("*.csv"))
    if csvs:
        DATA_PATH = csvs[0]
        print("Using found CSV:", DATA_PATH.resolve())
    else:
        raise FileNotFoundError(
            f"CSV not found at {DATA_PATH.resolve()}. Download the dataset from the links above, "
            "put it in dataset/, and update DATA_PATH."
        )

raw = pd.read_csv(DATA_PATH)
raw.columns = [str(c).strip().lower().replace(" ", "_").replace("-", "_") for c in raw.columns]
print("Raw shape:", raw.shape)
display(raw.head())

def find_col(options):
    return next((c for c in raw.columns if c in options), None)

colmap = {
    "date": find_col(["date", "datetime", "timestamp"]),
    "open": find_col(["open", "opening_price"]),
    "high": find_col(["high", "highest_price"]),
    "low": find_col(["low", "lowest_price"]),
    "close": find_col(["close", "closing_price", "adj_close", "adjclose"]),
    "volume": find_col(["volume", "vol"]),
    "ticker": find_col(["name", "symbol", "ticker", "company", "stock"])
}
if colmap["date"] is None or colmap["close"] is None:
    raise ValueError(f"Could not find Date and Close columns. Available columns: {list(raw.columns)}")

df = raw.rename(columns={v:k for k,v in colmap.items() if v is not None}).copy()
if "ticker" in df.columns:
    tickers = sorted(df["ticker"].dropna().astype(str).unique())
    chosen = str(TICKER_FILTER) if TICKER_FILTER is not None else (tickers[0] if tickers else None)
    if chosen is not None:
        if chosen not in tickers:
            raise ValueError(f"Ticker {chosen!r} not found. Example values: {tickers[:15]}")
        df = df[df["ticker"].astype(str) == chosen].copy()
        print("Selected ticker/company:", chosen)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
for c in ["open", "high", "low", "close", "volume"]:
    if c in df:
        df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["date", "close"]).sort_values("date")
df = df.drop_duplicates(subset=["date"], keep="last").reset_index(drop=True)
if len(df) < 100:
    raise ValueError(f"Only {len(df)} usable observations; please use a larger dataset.")
print("Clean data shape:", df.shape)
print("Date range:", df.date.min().date(), "to", df.date.max().date())
display(df.head())

## 3. Data understanding and quality checks

In [ ]:
print("Shape:", df.shape)
display(df.dtypes.to_frame("Data type"))
display(df.isna().sum().to_frame("Missing values"))
print("Duplicate rows:", df.duplicated().sum())
display(df.select_dtypes(include=np.number).describe().T)

## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(11,4))
plt.plot(df.date, df.close)
plt.title("Closing Price Over Time"); plt.xlabel("Date"); plt.ylabel("Close")
plt.tight_layout(); plt.savefig("figures/closing_price_over_time.png", dpi=150); plt.show()

if "volume" in df:
    plt.figure(figsize=(11,4))
    plt.plot(df.date, df.volume, linewidth=.8)
    plt.title("Trading Volume Over Time"); plt.xlabel("Date"); plt.ylabel("Volume")
    plt.tight_layout(); plt.savefig("figures/volume_over_time.png", dpi=150); plt.show()

plt.figure(figsize=(8,4))
plt.hist(df.close, bins=40)
plt.title("Distribution of Closing Prices"); plt.xlabel("Close"); plt.ylabel("Frequency")
plt.tight_layout(); plt.savefig("figures/closing_price_distribution.png", dpi=150); plt.show()

num_cols = [c for c in ["open","high","low","close","volume"] if c in df]
if len(num_cols) > 1:
    corr = df[num_cols].corr()
    plt.figure(figsize=(6,5))
    plt.imshow(corr, vmin=-1, vmax=1, aspect="auto")
    plt.xticks(range(len(num_cols)), num_cols, rotation=45, ha="right")
    plt.yticks(range(len(num_cols)), num_cols)
    plt.colorbar(label="Correlation")
    plt.title("OHLCV Correlation Heatmap")
    plt.tight_layout(); plt.savefig("figures/correlation_heatmap.png", dpi=150); plt.show()

## 5. Feature engineering and target

In [ ]:
data = df.copy()
data["daily_return"] = data.close.pct_change()
data["log_return"] = np.log(data.close / data.close.shift(1))
for w in [7, 14, 30]:
    data[f"sma_{w}"] = data.close.rolling(w).mean()
    data[f"volatility_{w}"] = data.daily_return.rolling(w).std()
data["momentum_5"] = data.close - data.close.shift(5)
data["sma_gap_7_30"] = data["sma_7"] - data["sma_30"]
for lag in [1, 2, 3, 5, 7]:
    data[f"close_lag_{lag}"] = data.close.shift(lag)
    data[f"return_lag_{lag}"] = data.daily_return.shift(lag)

# Target: next observed trading day's closing price
data["target_next_close"] = data.close.shift(-1)
feature_cols = [c for c in data.select_dtypes(include=np.number).columns
                if c not in ["target_next_close"]]
model_data = data.replace([np.inf, -np.inf], np.nan).dropna(subset=feature_cols + ["target_next_close"]).copy()

print("Number of engineered features:", len(feature_cols))
print(feature_cols)
print("Rows ready for modelling:", len(model_data))
display(model_data[["date", "close", "target_next_close"] + feature_cols[:5]].head())

plt.figure(figsize=(11,4))
plt.plot(data.date, data.close, label="Close")
plt.plot(data.date, data.sma_7, label="7-day SMA")
plt.plot(data.date, data.sma_30, label="30-day SMA")
plt.title("Closing Price and Moving Averages"); plt.xlabel("Date"); plt.ylabel("Price"); plt.legend()
plt.tight_layout(); plt.savefig("figures/moving_averages.png", dpi=150); plt.show()

plt.figure(figsize=(11,4))
plt.plot(data.date, data.volatility_14)
plt.title("14-Day Rolling Volatility"); plt.xlabel("Date"); plt.ylabel("Return standard deviation")
plt.tight_layout(); plt.savefig("figures/rolling_volatility.png", dpi=150); plt.show()

## 6. Chronological train / validation / test split

In [ ]:
n = len(model_data)
train_end = int(n * (1 - VALIDATION_FRACTION - TEST_FRACTION))
val_end = int(n * (1 - TEST_FRACTION))
train = model_data.iloc[:train_end].copy()
val = model_data.iloc[train_end:val_end].copy()
test = model_data.iloc[val_end:].copy()

for name, part in [("Train", train), ("Validation", val), ("Test", test)]:
    print(f"{name}: {len(part)} rows | {part.date.min().date()} to {part.date.max().date()}")

X_train, y_train = train[feature_cols], train.target_next_close
X_val, y_val = val[feature_cols], val.target_next_close
X_test, y_test = test[feature_cols], test.target_next_close

## 7. Train baseline and machine-learning models

In [ ]:
predictions = {}
# Naive benchmark: tomorrow's price equals today's close
predictions["Naive (last close)"] = test.close.to_numpy()

linear = Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())])
linear.fit(X_train, y_train)
predictions["Linear Regression"] = linear.predict(X_test)

rf = RandomForestRegressor(n_estimators=250, min_samples_leaf=2,
                           random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
predictions["Random Forest"] = rf.predict(X_test)

try:
    from xgboost import XGBRegressor
    boosting = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=.04,
                             subsample=.85, colsample_bytree=.85,
                             objective="reg:squarederror", random_state=RANDOM_STATE,
                             n_jobs=-1)
    boosting_name = "XGBoost"
except ImportError:
    from sklearn.ensemble import HistGradientBoostingRegressor
    boosting = HistGradientBoostingRegressor(max_iter=250, learning_rate=.05,
                                              max_leaf_nodes=20, random_state=RANDOM_STATE)
    boosting_name = "HistGradientBoosting (XGBoost fallback)"
boosting.fit(X_train, y_train)
predictions[boosting_name] = boosting.predict(X_test)
print("Models trained:", list(predictions))

## 8. Optional LSTM model

In [ ]:
# Optional deep-learning model; the main sklearn models work without TensorFlow.
try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    from sklearn.preprocessing import MinMaxScaler

    seq_cols = ["close", "daily_return", "sma_7", "sma_30", "volatility_14"]
    seq = data[["date"] + seq_cols + ["target_next_close"]].dropna().reset_index(drop=True)
    train_cut, val_cut = train.date.max(), val.date.max()
    scaler_x, scaler_y = MinMaxScaler(), MinMaxScaler()
    scaler_x.fit(seq.loc[seq.date <= train_cut, seq_cols])
    scaler_y.fit(seq.loc[seq.date <= train_cut, ["target_next_close"]])
    xs = scaler_x.transform(seq[seq_cols])
    ys = scaler_y.transform(seq[["target_next_close"]]).ravel()
    window = 20
    Xs, Ys, dates = [], [], []
    for i in range(window, len(seq)):
        Xs.append(xs[i-window:i]); Ys.append(ys[i]); dates.append(seq.date.iloc[i])
    Xs, Ys, dates = np.array(Xs), np.array(Ys), pd.to_datetime(dates)
    tr = dates <= train_cut
    va = (dates > train_cut) & (dates <= val_cut)
    te = dates > val_cut
    tf.random.set_seed(RANDOM_STATE)
    lstm = Sequential([LSTM(32, input_shape=(Xs.shape[1], Xs.shape[2])),
                       Dropout(.2), Dense(16, activation="relu"), Dense(1)])
    lstm.compile(optimizer="adam", loss="mse")
    lstm.fit(Xs[tr], Ys[tr], validation_data=(Xs[va], Ys[va]), epochs=40,
             batch_size=32, callbacks=[EarlyStopping(monitor="val_loss", patience=6,
             restore_best_weights=True)], verbose=0)
    lstm_pred = scaler_y.inverse_transform(lstm.predict(Xs[te], verbose=0)).ravel()
    lstm_actual = seq.loc[dates[te].isin(seq.date), "target_next_close"].to_numpy()
    lstm_dates = dates[te]
    predictions["LSTM"] = lstm_pred
    print("LSTM trained successfully.")
except ImportError:
    print("TensorFlow not installed; LSTM skipped. Install with: pip install tensorflow")
except Exception as e:
    print("LSTM skipped due to a data/runtime issue:", repr(e))

## 9. Model evaluation

In [ ]:
def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    mape = np.mean(np.abs((y-p) / np.maximum(np.abs(y), 1e-8))) * 100
    return {
        "MAE": mean_absolute_error(y, p),
        "RMSE": np.sqrt(mean_squared_error(y, p)),
        "MAPE (%)": mape,
        "R2": r2_score(y, p),
    }

rows = []
for name, pred in predictions.items():
    if name == "LSTM":
        # Align LSTM test observations by date
        mask = test.date.isin(lstm_dates)
        actual = test.loc[mask, "target_next_close"].to_numpy()
        current = test.loc[mask, "close"].to_numpy()
        if len(actual) != len(pred):
            # If dataset window alignment differs, evaluate only the shared number of final observations
            k = min(len(actual), len(pred))
            actual, current, pred = actual[-k:], current[-k:], pred[-k:]
    else:
        actual, current = y_test.to_numpy(), test.close.to_numpy()
    row = {"Model": name, **metrics(actual, pred),
           "Directional Accuracy (%)": np.mean((pred > current) == (actual > current)) * 100}
    rows.append(row)

results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
display(results)
results.to_csv("results/model_comparison.csv", index=False)

## 10. Visualize predictions and errors

In [ ]:
# Plot models that share the test split
plt.figure(figsize=(12,5))
plt.plot(test.date, y_test, label="Actual", linewidth=2)
for name, pred in predictions.items():
    if name != "LSTM" and len(pred) == len(test):
        plt.plot(test.date, pred, label=name, alpha=.8)
plt.title("Actual vs Predicted Next-Day Closing Price")
plt.xlabel("Date"); plt.ylabel("Closing price"); plt.legend()
plt.tight_layout(); plt.savefig("figures/actual_vs_predicted.png", dpi=150); plt.show()

best_name = results.iloc[0]["Model"]
best_pred = predictions[best_name]
if len(best_pred) == len(test):
    residuals = y_test.to_numpy() - best_pred
    plt.figure(figsize=(8,4))
    plt.hist(residuals, bins=35)
    plt.title(f"Residual Distribution — {best_name}")
    plt.xlabel("Actual - Predicted"); plt.ylabel("Frequency")
    plt.tight_layout(); plt.savefig("figures/residual_distribution.png", dpi=150); plt.show()
    errors = pd.DataFrame({"date": test.date, "actual_next_close": y_test,
                           "predicted": best_pred, "error": residuals,
                           "absolute_error": np.abs(residuals)})
    display(errors.sort_values("absolute_error", ascending=False).head(10))
    errors.to_csv("results/test_predictions_and_errors.csv", index=False)

## 11. Feature importance

In [ ]:
importance = pd.DataFrame({"Feature": feature_cols, "Importance": rf.feature_importances_})
importance = importance.sort_values("Importance", ascending=False).head(15)
display(importance)
plt.figure(figsize=(9,5))
plt.barh(importance.Feature[::-1], importance.Importance[::-1])
plt.title("Top Random Forest Feature Importances"); plt.xlabel("Importance")
plt.tight_layout(); plt.savefig("figures/feature_importance.png", dpi=150); plt.show()

## 12. Conference learning integration

| Conference concept | Implementation |
|---|---|
| Trustworthy AI | Chronological split, leakage prevention, transparent metrics, error analysis |
| Sustainable AI | Compare model complexity; avoid unnecessary complexity; optional early stopping |
| Signal processing | Returns, volatility, moving averages, momentum and lag features |
| Domain-specific AI | Financial time-series features and next-trading-observation target |
| Continuous monitoring | Save test errors and discuss drift and recalibration |

The original project report proposes GAN-based sequence generation and reinforcement-learning adaptation as an advanced extension. This notebook prioritizes a reproducible model pipeline; it does not claim GAN/RL results unless those components are separately implemented and tested.

## 13. Results interpretation and limitations

In [ ]:
display(results)
print("Best model by test RMSE:", results.iloc[0]["Model"])
print("Discuss whether the selected model improves on the naive benchmark.")
print("Review the actual-vs-predicted plot and residual distribution.")
print("Financial prices are uncertain and influenced by events absent from historical data.")
print("Historical model performance does not guarantee future performance. This is not financial advice.")

## 14. GitHub submission checklist

- `predictive_financial_forecasting.ipynb`
- `dataset/` with the CSV, if its license allows redistribution
- `README.md`
- `requirements.txt`
- `figures/` generated by notebook
- `results/` generated by notebook

Do not commit API keys or credentials. If the dataset license restricts redistribution, provide download instructions instead of uploading the raw data.